## Паплайн для загрузки данных, предобработки, генерации прогнозов и расчета метрик

Здесь боевой пайплайн загрузки данных и генерации прогнозов, но в конце идет расчет метрик по прогнзам двух моделей rf и cb относительно истинных значений, расставленных по словарю articles (который в свою очередь был размечен вручную экспертами и пополняется за счет фидбэка клиентов).

Ниже можно выгрузить срез платежей за любой период, получить по нему прогнозы модели, далее сгенерировать истинные значения на базе размеченного словаря, потом отбросить строки, в которых статьи не сошлись со словарными (новые еще не размеченные статьи) и посчитать метрики.

**Вывод**: платежи, которые имеют метки статей аналогичные словарю по которому размечалась тренировочная выборка - показывают высокие метрики, потому что модель де-факто работает как эвристика. Для тестирования обощающей способности, нужно пробовать на платежах с "новыми" статьями, которых модель не видела. 

In [2]:
import pandas as pd
import numpy as np
import random
import os
import re
import requests

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import IncrementalPCA

# ⏳ прогресс-бары
import tqdm as tqdm_lib
from tqdm import tqdm

# 🧠 обработка текста и NLP
import spacy
try:
    import transformers
    from transformers import AutoModel, AutoTokenizer
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "transformers"], stdout=subprocess.DEVNULL)
    import transformers
    from transformers import AutoModel, AutoTokenizer    
    
# 🤖 pyTorch
import torch

# загрузка моделей и функци для предобработки данных
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report,roc_auc_score
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.preprocessing import label_binarize

from imblearn.combine import SMOTEENN
from imblearn.under_sampling import RandomUnderSampler

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
import joblib

import catboost
from catboost import CatBoostClassifier, utils


pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.width', 0)
pd.set_option('display.max_colwidth', 120)

os.environ["TOKENIZERS_PARALLELISM"] = "false"

/opt/anaconda3/envs/datasphere_cli_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
RANDOM_STATE = 42
torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

In [4]:
# создаем функцию очистки текста
def _clean_single_text(text):
    return re.sub(r"[^\w\s]", " ", text.lower())

# создаем функцию предобработки текста
def preprocess_texts_optimized(texts, nlp_model_name,
                               batch_size_cpu=256,
                               num_processes_for_cleaning=-1,
                               num_processes_for_spacy_cpu=-1):
    
    print(f"🔍 Запуск предобработки для {len(texts)} текстов...")
    
    # предварительная очистка текстов
    cleaned_texts = [_clean_single_text(text) for text in tqdm(texts, desc="Очистка")]

    # spaCy и лемматизация
    nlp = None
    processed_lemmas = []
    
    # загрузка NLP модели
    print(f"🔍 Загрузка spaCy модели: '{nlp_model_name}'.")
    nlp = spacy.load(nlp_model_name)

    # используем n_process для параллелизации
    if num_processes_for_spacy_cpu == -1:
        cpu_count = os.cpu_count()
        num_processes_for_spacy_cpu = max(1, cpu_count - 1)
    
    print(f"🔍 Лемматизация будет выполнена в {num_processes_for_spacy_cpu} потоках.")
    
    for doc in tqdm(nlp.pipe(cleaned_texts, batch_size=batch_size_cpu, n_process=num_processes_for_spacy_cpu), total=len(cleaned_texts), desc="Лемматизация (CPU)"):
        lemmas = [token.lemma_ for token in doc]
        processed_lemmas.append(" ".join(lemmas))
    
    print(f"🔍 Предобработка завершена. Обработано {len(processed_lemmas)} текстов.")
    return processed_lemmas

# создаем функцию для получения усредненного эмбеддинга текста
def get_embeddings_batch(texts, tokenizer, model, device, batch_size=64):
    texts = list(texts)
    embeddings = []

    print(f"🔍 Начало генерации эмбеддингов для {len(texts)} текстов на устройстве '{device}'.")
    for i in tqdm(range(0, len(texts), batch_size), desc="Генерация эмбеддингов"):
        batch_texts = texts[i:i+batch_size]

        inputs = tokenizer(
            batch_texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=512
        )

        # Переносим каждый тензор на устройство
        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = model(**inputs)

        # берем attention mask (1 — реальные токены, 0 — паддинг)
        mask = inputs["attention_mask"].unsqueeze(-1).expand(outputs.last_hidden_state.size())
        masked_embeddings = outputs.last_hidden_state * mask

        # считаем среднее только по непаддинговым токенам
        summed = masked_embeddings.sum(dim=1)
        counts = mask.sum(dim=1)
        mean_pooled = summed / counts

        embeddings.extend(mean_pooled.cpu().numpy())
    
    print(f"🔍 Генерация эмбеддингов завершена")
    
    return embeddings

# функция предобработки входного датасета в целом
def prepare_data(data, is_train, scaler=None, ipca=None):
    
    data = data.drop(
            [
                "robot_id",
                "accounts__id",
                "articles__id",
                "articles__user_id",
                "projects__id",
                "projects__user_id",
                "counterparties__id",
                "counterparties__user_id",
                "robots__user_id",
                "article_alternative_names__user_id",
            ],
            axis=1,
        )

    # поправим типы данных и заполним пропуски метками missing (для текстовых значений категорий) и 0 для пропущенных ID
    data[
        [
            "articles__parent_id",
            "projects__parent_id",
            "counterparties__parent_id",
            "robots__id",
        ]
    ] = (
        data[
            [
                "articles__parent_id",
                "projects__parent_id",
                "counterparties__parent_id",
                "robots__id",
            ]
        ]
        .fillna(0)
        .astype("int64")
    )

    data["purpose"] = data["purpose"].fillna("missing")
    data["articles__name"] = data["articles__name"].fillna("missing")
    data["projects__name"] = data["projects__name"].fillna("missing")
    data["counterparties__name"] = data["counterparties__name"].fillna("missing")

    # конвертируем дату в datetime
    data["date"] = pd.to_datetime(data["date"], yearfirst=True, errors='coerce')

    # и убираем записи из будущего и меньше нуля (и такое бывает)
    yesterday = pd.Timestamp.today().normalize() - pd.Timedelta(days=1)
    data = data[data["date"] <= yesterday]
    #data = data[data["payments_amount"] > 0]

    # переименуем и поправим тип столбца с фондами
    data = data.rename(columns={"accounts__user_id": "user_id"})
    data["user_id"] = data["user_id"].fillna(0).astype("int64")

    # кодируем текстовые поля
    # сначала очищаем и лемматизируем тексты
    data["clean_purpose"] = preprocess_texts_optimized(texts=data["purpose"],nlp_model_name="ru_core_news_sm")

    # грузим модели 
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    tokenizer = AutoTokenizer.from_pretrained("DeepPavlov/rubert-base-cased")
    model = AutoModel.from_pretrained("DeepPavlov/rubert-base-cased")
    model = model.to(device)
    model.eval()
    
    # и запускаем генерацию эмбеддингов в назначении платежа
    data["purpose_emb"] = get_embeddings_batch(data["clean_purpose"], tokenizer, model, device)

    # 1. усредняем эмбеддинг в одно число
    data["purpose_mean"] = data["purpose_emb"].apply(lambda x: float(np.mean(x)))

    # 2. выделяем три главные компоненты с предварительным масштабированием и по батчам  
    batch_size = 10_000
    
    if is_train:
        scaler = StandardScaler()
        ipca = IncrementalPCA(n_components=3)

        # обучаем скейлер
        for i in tqdm(range(0, len(data), batch_size), desc="Обучение StandardScaler"):
            batch = np.vstack(data["purpose_emb"].iloc[i:i+batch_size])
            scaler.partial_fit(batch)

        # обучаем PCA на масштабированных данных
        for i in tqdm(range(0, len(data), batch_size), desc="Обучение IncrementalPCA"):
            batch = np.vstack(data["purpose_emb"].iloc[i:i+batch_size])
            batch_scaled = scaler.transform(batch)
            ipca.partial_fit(batch_scaled)

    # применяем трансформацию ко всему массиву
    transformed_batches = []
    for i in tqdm(range(0, len(data), batch_size), desc="Применяем PCA к эмбеддингам"):
        batch = np.vstack(data["purpose_emb"].iloc[i:i+batch_size]).astype(np.float32)
        batch_scaled = scaler.transform(batch)
        transformed_batches.append(ipca.transform(batch_scaled))
        
    purpose_pca_features = np.vstack(transformed_batches)

    # делаем датафрейм
    pca_column_names = [f"purpose_pca_{i+1}" for i in range(3)]
    data[pca_column_names] = purpose_pca_features

    # удалим ненужные столбцы
    data.drop(columns=["purpose", "clean_purpose", "purpose_emb"], inplace=True)

    # генерируем эмбеддинги для названий статей
    # сначала очищаем и лемматизируем тексты
    data["clean_articles__name"] = preprocess_texts_optimized(texts=data["articles__name"],nlp_model_name="ru_core_news_sm")
     # и запускаем генерацию эмбеддингов в названии статей
    data["articles__name_emb"] = get_embeddings_batch(data["clean_articles__name"], tokenizer, model, device)
    # усредняем эмбеддинг в одно число
    data["articles__name_mean"] = data["articles__name_emb"].apply(lambda x: float(np.mean(x)))
    # удалим ненужные столбцы
    data.drop(columns=["articles__name", "clean_articles__name", "articles__name_emb"], inplace=True)

    # генерируем эмбеддинги для названий проектов
    # сначала очищаем и лемматизируем тексты
    data["clean_projects__name"] = preprocess_texts_optimized(texts=data["projects__name"],nlp_model_name="ru_core_news_sm")
     # и запускаем генерацию эмбеддингов в названии статей
    data["projects__name_emb"] = get_embeddings_batch(data["clean_projects__name"], tokenizer, model, device)
    # усредняем эмбеддинг в одно число
    data["projects__name_mean"] = data["projects__name_emb"].apply(lambda x: float(np.mean(x)))
    # удалим ненужные столбцы
    data.drop(columns=["projects__name", "clean_projects__name", "projects__name_emb"], inplace=True)
    
    # генерируем эмбеддинги для названий доноров
    # сначала очищаем и лемматизируем тексты
    data["clean_counterparties__name"] = preprocess_texts_optimized(texts=data["counterparties__name"],nlp_model_name="ru_core_news_sm")
     # и запускаем генерацию эмбеддингов в названии статей
    data["counterparties__name_emb"] = get_embeddings_batch(data["clean_counterparties__name"], tokenizer, model, device)
    # усредняем эмбеддинг в одно число
    data["counterparties__name_mean"] = data["counterparties__name_emb"].apply(lambda x: float(np.mean(x)))
    # удалим ненужные столбцы
    data.drop(columns=["counterparties__name", "clean_counterparties__name", "counterparties__name_emb"], inplace=True)


    # сбрасываем неактуальные столбцы
    data = data.drop(columns=[ 
        'date', 'expenditure',
        'payments_amount','user_id','account_id', 
        'contractor_id', 'article_id', 'project_id', 
        'counterpartie_id', 'donor_id', 'donor_cat_id',
        'articles__parent_id', 'projects__parent_id',
        'counterparties__parent_id', 'robots__id'])

    return data,scaler,ipca


### Загрузим и подготовим данные для прогноза

In [166]:
# загружаем новые платежи за вчера

url_down = "https://api.lemonpie.tech/api/payments/ai"
headers = {"Authorization": "Bearer YOUR_API_TOKEN"}

#start_date = str((pd.Timestamp.today().normalize() - pd.Timedelta(days=1)).date())
#end_date = start_date #str(pd.Timestamp.today().normalize().date())
start_date = "2025-10-10"
end_date = "2025-10-20"


params = {
    "limit": 5000,
    "page": 1,
    "expenditure": "incoming",
    "date_from": start_date,
    "date_to": end_date  
}

all_data = []

with tqdm(desc="Загружено страниц", unit=" стр", dynamic_ncols=True) as pbar:
    while True:
        response = requests.get(url_down, headers=headers, params=params)
        if response.status_code != 200:
            print(f"❌ Ошибка загрузки данных с сервера: {response.status_code}")
            break

        result = response.json()
        page_data = result.get("data", [])
        if not page_data:
            break
        
        all_data.extend(page_data)

        params["page"] += 1
        pbar.update(1)
        
# преобразуем в таблицу (вложенные поля будут с __)
data_full = pd.json_normalize(all_data, sep="__")
print(f"ℹ️ Данные загружены с сервера. Количество записей: {len(data_full)}")
#data_full.to_csv("data_download.csv", index=False)

Загружено страниц: 2 стр [00:05,  2.87s/ стр]

ℹ️ Данные загружены с сервера. Количество записей: 8901


In [167]:

print('Проверим пропуски по основным признакам:',
    (data_full[['purpose','articles__name','projects__name','counterparties__name']].isna().sum()))
print(
    "Количество строк, где все 3 дополнительных признака отсутствуют:",
    data_full[['articles__name','projects__name','counterparties__name']]
        .isna()
        .all(axis=1)
        .sum()
)
# сбросим строки в которых все 3 дополнительных поля отсутствуют (роботы не отработали - качество прогноза будет плохое)
data_full = data_full.dropna(subset=['articles__name', 'projects__name', 'counterparties__name'], how='all')


Проверим пропуски по основным признакам: purpose                  26
articles__name          383
projects__name          794
counterparties__name    604
dtype: int64
Количество строк, где все 3 дополнительных признака отсутствуют: 318


In [168]:
data_full.head(3)

,id,account_id,contractor_id,date,payments_amount,purpose,article_id,expenditure,project_id,counterpartie_id,donor_id,robot_id,donor_cat_id,accounts__id,accounts__user_id,articles__id,articles__user_id,articles__parent_id,articles__name,projects__id,projects__user_id,projects__parent_id,projects__name,counterparties__id,counterparties__user_id,counterparties__parent_id,counterparties__name,robots__id,robots__user_id,article_alternative_names__user_id,uc__uc_id
0,1103275,2708,46,2025-10-10,482.00,"Перевод средств по договору № от 19.01.2023 по Реестру Операций от 09.10.2025. Сумма комиссии 18 руб. 00 коп., НДС ...",40640,incoming,3705,0,0,8051,9792,2708,954,40640.0,954.0,40639.0,CloudPayments,3705.0,954.0,0.0,Уставная деятельность,9792.0,954.0,9789.0,...массовые,8051.0,954.0,NaN,None
2,1105223,2367,1807,2025-10-10,1009.33,Выплата процентов по депозитной сделке № UNV-0000006958989 от 08.10.2025 за период с 09.10.2025 по 10.10.2025 соглас...,39896,incoming,3509,0,0,7502,0,2367,942,39896.0,942.0,39895.0,% на остаток по счету,3509.0,942.0,0.0,Уставные деятельность,NaN,NaN,NaN,None,7502.0,942.0,NaN,None
3,1105225,2367,96992,2025-10-10,1473625.56,Возврат средств по депозитной сделке № UNV-0000006958989 от 08.10.2025 согласно Правилам банковского обслуживания. Н...,39927,incoming,3514,0,0,7478,0,2367,942,39927.0,942.0,39925.0,Перевод депозита,3514.0,942.0,0.0,Ы_Возвраты,NaN,NaN,NaN,None,7478.0,942.0,NaN,None


In [169]:
# предобрабатываем датасет
    
scaler_emb_path = 'scaler.pkl'
ipca_emb_path = 'ipca.pkl'

scaler = joblib.load(scaler_emb_path)
ipca = joblib.load(ipca_emb_path)

data_full_prepared, _, _ = prepare_data(data_full, is_train=False, scaler=scaler, ipca=ipca)


🔍 Запуск предобработки для 8583 текстов...


Очистка: 100%|██████████| 8583/8583 [00:00<00:00, 343256.49it/s]

🔍 Загрузка spaCy модели: 'ru_core_news_sm'.


🔍 Лемматизация будет выполнена в 7 потоках.


Лемматизация (CPU): 100%|██████████| 8583/8583 [00:28<00:00, 300.42it/s] 


...

🔍 Загрузка spaCy модели: 'ru_core_news_sm'.
🔍 Лемматизация будет выполнена в 7 потоках.


Лемматизация (CPU): 100%|██████████| 8583/8583 [00:22<00:00, 381.84it/s] 


🔍 Предобработка завершена. Обработано 8583 текстов.
🔍 Начало генерации эмбеддингов для 8583 текстов на устройстве 'cpu'.


Генерация эмбеддингов:   0%|          | 0/135 [00:00<?, ?it/s]/opt/anaconda3/envs/datasphere_cli_env/lib/python3.10/site-packages/torch/nn/modules/module.py:1750: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)
Генерация эмбеддингов: 100%|██████████| 135/135 [00:21<00:00,  6.24it/s]

🔍 Генерация эмбеддингов завершена


In [170]:
data_full_prepared.head()

,id,uc__uc_id,purpose_mean,purpose_pca_1,purpose_pca_2,purpose_pca_3,articles__name_mean,projects__name_mean,counterparties__name_mean
0,1103275,None,0.000695,12.350216,-2.595701,-19.962817,-0.001473,0.000916,-0.000312
2,1105223,None,-0.001364,9.254664,4.539226,-10.099314,0.001054,-0.001267,-0.000798
3,1105225,None,-0.001025,9.523563,6.560456,-5.622940,0.000025,-0.001713,-0.000798
4,1105226,None,0.000197,12.413155,-2.688546,-18.877105,-0.001473,-0.001267,-0.000312
5,1105227,None,0.000700,11.456743,-1.660316,-18.216598,-0.001473,-0.001267,-0.000312


### Генерируем прогнозы

#### randomforest

In [171]:
best_model_rf = RandomForestClassifier()
best_model_rf = joblib.load("model_rf.pkl")
print("Параметры загруженной модели:", best_model_rf.get_params())

features = [
    'purpose_mean', 'purpose_pca_1', 'purpose_pca_2', 'purpose_pca_3',
    'articles__name_mean', 'projects__name_mean', 'counterparties__name_mean'
]

data_full_prepared['uc__uc_id_rf'] = best_model_rf.predict(data_full_prepared[features])


Параметры загруженной модели: {'bootstrap': True, 'ccp_alpha': 0.0, 'class_weight': 'balanced', 'criterion': 'gini', 'max_depth': 9, 'max_features': 'sqrt', 'max_leaf_nodes': None, 'max_samples': None, 'min_impurity_decrease': 0.0, 'min_samples_leaf': 1, 'min_samples_split': 2, 'min_weight_fraction_leaf': 0.0, 'n_estimators': 1000, 'n_jobs': -1, 'oob_score': False, 'random_state': 42, 'verbose': 2, 'warm_start': False}


[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  25 tasks      | elapsed:    0.0s
[Parallel(n_jobs=8)]: Done 146 tasks      | elapsed:    0.0s
[Parallel(n_jobs=8)]: Done 349 tasks      | elapsed:    0.1s
[Parallel(n_jobs=8)]: Done 632 tasks      | elapsed:    0.1s
[Parallel(n_jobs=8)]: Done 1000 out of 1000 | elapsed:    0.1s finished


In [172]:
data_full_prepared.head()

,id,uc__uc_id,purpose_mean,purpose_pca_1,purpose_pca_2,purpose_pca_3,articles__name_mean,projects__name_mean,counterparties__name_mean,uc__uc_id_rf
0,1103275,None,0.000695,12.350216,-2.595701,-19.962817,-0.001473,0.000916,-0.000312,1
2,1105223,None,-0.001364,9.254664,4.539226,-10.099314,0.001054,-0.001267,-0.000798,7
3,1105225,None,-0.001025,9.523563,6.560456,-5.622940,0.000025,-0.001713,-0.000798,4
4,1105226,None,0.000197,12.413155,-2.688546,-18.877105,-0.001473,-0.001267,-0.000312,1
5,1105227,None,0.000700,11.456743,-1.660316,-18.216598,-0.001473,-0.001267,-0.000312,1


#### catboost

In [173]:
best_model_cb = CatBoostClassifier()
best_model_cb.load_model("model_cb.cbm")
print('Параметры загруженной модели: ', best_model_cb.get_params())

features = [
    'purpose_mean', 'purpose_pca_1', 'purpose_pca_2', 'purpose_pca_3',
    'articles__name_mean', 'projects__name_mean', 'counterparties__name_mean'
]

data_full_prepared['uc__uc_id_cb'] = best_model_cb.predict(data_full_prepared[features])


Параметры загруженной модели:  {'random_strength': 5, 'border_count': 256, 'od_wait': 10, 'verbose': 100, 'iterations': 1000, 'auto_class_weights': 'Balanced', 'loss_function': 'MultiClass', 'l2_leaf_reg': 9, 'task_type': 'GPU', 'depth': 6, 'min_data_in_leaf': 1, 'learning_rate': 0.05, 'random_seed': 42}


In [174]:
diff = data_full_prepared[data_full_prepared['uc__uc_id_rf'] != data_full_prepared['uc__uc_id_cb']]
diff.id.count()

830

Считаем истинные значения по размеченному словарю

In [8]:
slovar = pd.read_csv('../../uc_data_labeling/articles_uc_added.csv')

In [176]:
data_full['articles__name'] = data_full['articles__name'].str.lower()

data_full = data_full.merge(
    slovar[['raw', 'uc_id']],
    left_on='articles__name',
    right_on='raw',
    how='left'
)

data_full.rename(columns={'uc_id': 'target'}, inplace=True)
data_full.drop(columns=['raw'], inplace=True)
data_full.head(3)

,id,account_id,contractor_id,date,payments_amount,purpose,article_id,expenditure,project_id,counterpartie_id,donor_id,robot_id,donor_cat_id,accounts__id,accounts__user_id,articles__id,articles__user_id,articles__parent_id,articles__name,projects__id,projects__user_id,projects__parent_id,projects__name,counterparties__id,counterparties__user_id,counterparties__parent_id,counterparties__name,robots__id,robots__user_id,article_alternative_names__user_id,uc__uc_id,target
0,1103275,2708,46,2025-10-10,482.00,"Перевод средств по договору № от 19.01.2023 по Реестру Операций от 09.10.2025. Сумма комиссии 18 руб. 00 коп., НДС ...",40640,incoming,3705,0,0,8051,9792,2708,954,40640.0,954.0,40639.0,cloudpayments,3705.0,954.0,0.0,Уставная деятельность,9792.0,954.0,9789.0,...массовые,8051.0,954.0,NaN,None,1.0
1,1105223,2367,1807,2025-10-10,1009.33,Выплата процентов по депозитной сделке № UNV-0000006958989 от 08.10.2025 за период с 09.10.2025 по 10.10.2025 соглас...,39896,incoming,3509,0,0,7502,0,2367,942,39896.0,942.0,39895.0,% на остаток по счету,3509.0,942.0,0.0,Уставные деятельность,NaN,NaN,NaN,None,7502.0,942.0,NaN,None,7.0
2,1105225,2367,96992,2025-10-10,1473625.56,Возврат средств по депозитной сделке № UNV-0000006958989 от 08.10.2025 согласно Правилам банковского обслуживания. Н...,39927,incoming,3514,0,0,7478,0,2367,942,39927.0,942.0,39925.0,перевод депозита,3514.0,942.0,0.0,Ы_Возвраты,NaN,NaN,NaN,None,7478.0,942.0,NaN,None,4.0


In [156]:
# чтобы проставлять вручную значения, которые не подставились по словарю статей articles
data_full.loc[data_full['id'] == 1181174, 'target'] = 2

In [179]:
data_full[data_full.target.isna()]['id'].count()

139

In [180]:
forecasts = data_full_prepared[['id','uc__uc_id_rf','uc__uc_id_cb']].merge(data_full[['id', 'target']], how='left', on='id')
forecasts['target'] = forecasts['target'].astype('Int64')
forecasts.head()

,id,uc__uc_id_rf,uc__uc_id_cb,target
0,1103275,1,1,1
1,1105223,7,7,7
2,1105225,4,4,4
3,1105226,1,1,1
4,1105227,1,1,1


In [181]:
# если дропаем nan в истинных значениях
forecasts.dropna(subset=['target'], inplace=True)

In [182]:
# метрики прогнозов rf
accuracy = accuracy_score(forecasts['target'], forecasts['uc__uc_id_rf'])
precision = precision_score(forecasts['target'], forecasts['uc__uc_id_rf'], average='weighted')
recall = recall_score(forecasts['target'], forecasts['uc__uc_id_rf'], average='macro')
f1 = f1_score(forecasts['target'], forecasts['uc__uc_id_rf'], average='weighted')

print("Accuracy:", accuracy)
print("Precision (weighted):", precision)
print("Recall (macro):", recall)
print("F1-score (weighted):", f1)

print("\nОтчет по классам:")
print(classification_report(forecasts['target'], forecasts['uc__uc_id_rf']))

Accuracy: 0.9126006631927996
Precision (weighted): 0.9655164465775576
Recall (macro): 0.8738855996202081
F1-score (weighted): 0.9347331371176069

Отчет по классам:
              precision    recall  f1-score   support

         1.0       0.98      0.93      0.96      5049
         2.0       1.00      0.90      0.94      2886
         3.0       0.59      0.53      0.56        57
         4.0       0.88      0.81      0.84       121
         5.0       0.49      0.84      0.62       122
         6.0       0.64      0.92      0.76       140
         7.0       0.25      0.95      0.40        55
         8.0       0.21      1.00      0.34        11
         9.0       0.01      1.00      0.02         3

    accuracy                           0.91      8444
   macro avg       0.56      0.87      0.60      8444
weighted avg       0.97      0.91      0.93      8444



In [183]:
# метрики прогнозов cb
accuracy = accuracy_score(forecasts['target'], forecasts['uc__uc_id_cb'])
precision = precision_score(forecasts['target'], forecasts['uc__uc_id_cb'], average='weighted')
recall = recall_score(forecasts['target'], forecasts['uc__uc_id_cb'], average='macro')
f1 = f1_score(forecasts['target'], forecasts['uc__uc_id_cb'], average='weighted')


print("Accuracy:", accuracy)
print("Precision (weighted):", precision)
print("Recall (macro):", recall)
print("F1-score (weighted):", f1)

print("\nОтчет по классам:")
print(classification_report(forecasts['target'], forecasts['uc__uc_id_cb']))

Accuracy: 0.9655376598768356
Precision (weighted): 0.9686930004243464
Recall (macro): 0.9598288845100463
F1-score (weighted): 0.9663813878354673

Отчет по классам:
              precision    recall  f1-score   support

         1.0       0.99      0.96      0.98      5049
         2.0       0.95      0.98      0.96      2886
         3.0       0.66      0.86      0.75        57
         4.0       0.95      0.94      0.95       121
         5.0       0.80      0.93      0.86       122
         6.0       0.82      0.97      0.89       140
         7.0       0.68      1.00      0.81        55
         8.0       0.92      1.00      0.96        11
         9.0       0.60      1.00      0.75         3

    accuracy                           0.97      8444
   macro avg       0.82      0.96      0.88      8444
weighted avg       0.97      0.97      0.97      8444



In [184]:
# помимо метрик глянем еще на распределения предсказаний (частоты классов)
print(forecasts['uc__uc_id_rf'].value_counts())
print(forecasts['uc__uc_id_cb'].value_counts())

1    4774
2    2590
9     248
5     209
7     207
6     201
4     111
8      53
3      51
Name: uc__uc_id_rf, dtype: int64
1    4873
2    2972
6     166
5     141
4     120
7      81
3      74
8      12
9       5
Name: uc__uc_id_cb, dtype: int64


In [185]:
# согласие моделей (agreement score)
agreement = (forecasts['uc__uc_id_rf'] == forecasts['uc__uc_id_cb']).mean()
print("Согласие моделей:", agreement)

Согласие моделей: 0.9041923259118901


In [ ]:
# сравним уверенности моделей на несовпадающих значениях

data_full_prepared['rf_max_prob'] = best_model_rf.predict_proba(data_full_prepared[features]).max(axis=1)
data_full_prepared['cb_max_prob'] = best_model_cb.predict_proba(data_full_prepared[features]).max(axis=1)

diff_probs = data_full_prepared.loc[diff.index, ['rf_max_prob','cb_max_prob']]
diff_probs